`https://drive.google.com/file/d/1Pu8TGBF_bFXbjPi4gJ864Q6IHFKhB2v3/view?usp=sharing`

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
adhamashraf202200953_technical_parsed_questions_path = kagglehub.dataset_download('adhamashraf202200953/technical-parsed-questions')

print('Data source import complete.')

Using Colab cache for faster access to the 'technical-parsed-questions' dataset.
Data source import complete.


In [ ]:
# Upgrade PyTorch to a unified CUDA 12.1 version
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install and upgrade all Hugging Face and evaluation libraries together
!pip install --upgrade transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score bert_score nltk

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
# !pip install --upgrade torchao

In [ ]:
import gc
import torch
import os
import json
import glob
import numpy as np
from datasets import Dataset
from transformers import (
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
DATA_FOLDER = "/root/.cache/kagglehub/datasets/adhamashraf202200953/technical-parsed-questions/versions/1"
OUTPUT_DIR = "./outputs/STABLE_QWEN"

BATCH_SIZE = 2
GRADIENT_ACC_STEPS = 16
MAX_LENGTH = 512

gc.collect()
torch.cuda.empty_cache()

In [ ]:
def migrate_essay_data(folder_path):
    all_essay_records = []
    search_pattern = os.path.join(folder_path, "*_essay.jsonl")
    essay_files = glob.glob(search_pattern)
    for file_path in essay_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        record = json.loads(line)
                        if record.get("type") == "essay" and record.get("model_answer"):
                            all_essay_records.append(record)
                    except: continue
    return all_essay_records

def preprocess_function(examples):
    full_texts = []
    for c, s, q, a in zip(examples["context"], examples["scenario_context"], examples["question"], examples["model_answer"]):
        text = f"Context: {c}\n\nScenario: {s} Question: {q} Answer: {a}{tokenizer.eos_token}"
        full_texts.append(text)

    model_inputs = tokenizer(full_texts, max_length=MAX_LENGTH, truncation=True)
    return model_inputs

In [ ]:
raw_essay_data = migrate_essay_data(DATA_FOLDER)
dataset = Dataset.from_list(raw_essay_data)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)

Map:   0%|          | 0/2209 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
print("Total training samples:", len(split_dataset["train"]))

Total training samples: 1988


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=5,
    learning_rate=2e-4,
    num_train_epochs=3,
    max_grad_norm=1.0,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,1.553000,1.560286
100,1.480941,1.522405
150,1.426780,1.509895
189,1.409159,1.506268


TrainOutput(global_step=189, training_loss=1.523026145955242, metrics={'train_runtime': 1944.2877, 'train_samples_per_second': 3.067, 'train_steps_per_second': 0.097, 'total_flos': 2.168953205419008e+16, 'train_loss': 1.523026145955242, 'epoch': 3.0})

In [ ]:
!zip -r "/content/QWEN-2.5-1.5B.zip" "/content/outputs"

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/STABLE_QWEN/ (stored 0%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/ (stored 0%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/training_args.bin (deflated 54%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/scaler.pt (deflated 64%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/rng_state.pth (deflated 26%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/adapter_config.json (deflated 60%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/chat_template.jinja (deflated 71%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/trainer_state.json (deflated 75%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/scheduler.pt (deflated 62%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/README.md (deflated 65%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/tokenizer_config.json (deflated 60%)
  adding: content/outputs/STABLE_QWEN/checkpoint-189/adapter_model.safetensors (deflat

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B"
QWEN_LORA_DIR = "/content/outputs/STABLE_QWEN/checkpoint-189"

print("Loading Qwen 1.5B...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, QWEN_LORA_DIR)

def ask_qwen(context_text):
    prompt = f"Context: {context_text}\n\nScenario:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    generated_answer = full_text[len(prompt):].strip()
    return f"Scenario: {generated_answer}"

Loading Qwen 1.5B...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
sample_context = """
When a Docker container exits immediately with a status code 0, it typically means the primary
foreground process specified in the ENTRYPOINT or CMD has completed its execution. Containers
require a persistent foreground process to remain active. To keep a debugging container running
without an active service, developers often append a blocking command like 'tail -f /dev/null'
or run the container interactively using the '-it' flags, though this can consume unnecessary
idle system resources.
"""
print(ask_qwen(sample_context))

Scenario: Your Docker container is used for monitoring and logging purposes, but it requires a persistent foreground process to keep running. Question: Propose a solution to keep your debugging container running without an active service, using the latest industry-standard practices. Explain the tradeoff. Answer: Use a blocking command like 'tail -f /dev/null' or run the container interactively using the '-it' flags. This keeps the container alive but consumes unnecessary idle system resources. A better approach would be to use a different lifecycle management tool or framework that provides more control over container lifetimes. Tradeoff: increased system resource usage vs. maintaining the debugging container running.
